# Langchain Begin - 01

In [ ]:
# Install required libraries
!pip install langchain transformers accelerate sentencepiece bitsandbytes langchain-community langchain-huggingface

In [ ]:
import os
from langchain_community.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

In [ ]:
model_name = "gpt2"

In [ ]:
# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

In [ ]:
# Create HuggingFace pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200,
    temperature=0.7,
    do_sample=True
)

In [ ]:
# Wrap into LangChain LLM
llm = HuggingFacePipeline(pipeline=pipe)

In [ ]:
# --- Example 1: Direct prompt ---
prompt = "Suggest me a skill that is in demand?"
response = llm.invoke(prompt)
print(" Suggested Skill:\n", response)

In [ ]:
# --- Example 2: PromptTemplate + Chain ---
template = "Give me 3 career skills that are in high demand in {year}."
prompt_template = PromptTemplate.from_template(template)
chain = prompt_template | llm | StrOutputParser()

response = chain.invoke({"year": "2025"})
print("\n Career Skills in 2025:\n", response)

# Langhcain Begin - 02

In [ ]:
from langchain_huggingface import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.memory import ConversationBufferMemory

In [ ]:
# Example 1: Simple Prompt Template
template = "You are a helpful assistant. User: {input}. Assistant:"
prompt = PromptTemplate(input_variables=["input"], template=template)
chain = LLMChain(llm=llm, prompt=prompt)

In [ ]:
# Run it
result = chain.invoke({"input": "Explain LangChain in one sentence."})
print(result['text'])
# Expected: A concise explanation.

In [ ]:
# Example 2: Add Memory for Chat History
memory = ConversationBufferMemory(input_key='input', memory_key='chat_history', return_messages=True)
chat_chain = LLMChain(llm=llm, prompt=prompt, memory=memory, verbose=True)
chat_chain.invoke({"input": "Hi, what's your name?"})
chat_chain.invoke({"input": "What did I just ask?"})
# Expected: Remembers previous message!